# ML-10 — Content Action Playbook

**Lane:** Refresh / Content Opportunity Scoring  
**Validated input:** Week-6 grouped-client audit of the five-feature Logistic Regression.

This notebook turns the model output into a **human-reviewed content action playbook**. The Week-6 audit found that the model's grouped-holdout AUROC was close to chance overall, even though its top-20 queue looked stronger. Therefore the score is used only as **exploratory decision-support and prioritization**. It is not an auto-edit, auto-delete, or causal recommendation engine.

All actions below are generated from the real 30,000-row starter dataset. No client names, URLs, or private queries are exported.


## 1. Ranked actions + reason codes

The playbook separates **priority** from **action**. A higher model score means “review earlier,” not “this page definitely needs a refresh.”

I use four practical archetypes:

| Archetype | Reason code | Action label | Human interpretation |
|---|---|---|---|
| Visible + low CTR | `LOW_CTR_VISIBLE` | `REVIEW_TITLE_SNIPPET` | Search visibility exists, but measured CTR is low. Review intent, title, snippet, SERP context, and cannibalization. |
| Stale + meaningful volume | `STALE_HIGH_VOLUME` | `REVIEW_REFRESH` | Page is old since last update and still receives meaningful impressions. Review factual freshness, coverage, links, and intent. |
| Visible + reasonable CTR | `VISIBLE_MONITOR` | `MONITOR` | Ranking/visibility is present without the clearest CTR warning. Do not force a refresh; monitor and inspect context. |
| Low evidence | `INSUFFICIENT_SIGNAL` | `HOLD` | Too little visibility or no valid position signal for a confident action queue. Collect more evidence first. |

**One primary reason code per page.** The rule gives precedence to low CTR at visible positions, then stale/high-volume pages, then visible-monitor pages.

### Decay / refresh insight

In this starter snapshot, pages with longer `days_since_last_update` can still have strong visibility, and recently updated pages can still decline. That means **staleness is a review trigger, not proof of decay**. A refresh recommendation should be confirmed by content context, intent, and measurement history before anyone edits the page.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

DATA_PATH = DATA_PATH.resolve()
REPO_ROOT = DATA_PATH.parents[2]

df = pd.read_csv(DATA_PATH)
df["decline_proxy"] = (df["trend_direction"] == "down").astype(int)  # evaluation only

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
]

# Fit the already-audited simple model on all starter rows only to produce
# an exploratory review-priority score for this non-production playbook.
model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    )),
])
model.fit(df[features], df["decline_proxy"])
df["review_priority_score"] = model.predict_proba(df[features])[:, 1]

# Transparent archetypes: no future-window or label-derived field is used.
valid_position = df["avg_position"] > 0
visible = valid_position & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)
low_ctr_visible = visible & (df["ctr"] < 0.50)
stale_high_volume = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 1000)
    & ~low_ctr_visible
)
visible_monitor = visible & ~low_ctr_visible & ~stale_high_volume

df["reason_code"] = np.select(
    [low_ctr_visible, stale_high_volume, visible_monitor],
    ["LOW_CTR_VISIBLE", "STALE_HIGH_VOLUME", "VISIBLE_MONITOR"],
    default="INSUFFICIENT_SIGNAL",
)
df["action_label"] = np.select(
    [low_ctr_visible, stale_high_volume, visible_monitor],
    ["REVIEW_TITLE_SNIPPET", "REVIEW_REFRESH", "MONITOR"],
    default="HOLD",
)

# Cost/value framing: score × log-scaled visibility, still only a triage aid.
visibility_value = np.log1p(df["impressions_90d"])
df["review_value_score"] = (
    100
    * df["review_priority_score"]
    * (visibility_value / visibility_value.max())
)

queue = (
    df.sort_values(
        ["review_value_score", "impressions_90d"],
        ascending=[False, False],
        kind="stable",
    )
    .reset_index(drop=True)
    .copy()
)
queue["rank"] = np.arange(1, len(queue) + 1)

print(f"Rows ranked: {len(queue):,}")
print(f"Observed decline-proxy rate: {df['decline_proxy'].mean():.1%}")
print("\nAction mix:")
display(
    queue.groupby(["action_label", "reason_code"], observed=True)
    .size()
    .rename("n")
    .reset_index()
    .sort_values("n", ascending=False)
)

display(queue[[
    "rank", "content_id", "review_value_score", "review_priority_score",
    "action_label", "reason_code", "impressions_90d", "ctr",
    "avg_position", "days_since_last_update"
]].head(15))


Rows ranked: 30,000
Observed decline-proxy rate: 54.2%

Action mix:


,action_label,reason_code,n
0,HOLD,INSUFFICIENT_SIGNAL,17971
3,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,9759
1,MONITOR,VISIBLE_MONITOR,2264
2,REVIEW_REFRESH,STALE_HIGH_VOLUME,6


,rank,content_id,review_value_score,review_priority_score,action_label,reason_code,impressions_90d,ctr,avg_position,days_since_last_update
0,1,content_c8e9d6ab9013,62.204363,0.668188,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,208678,0.00,9.7,104
1,2,content_36ff89c8214e,60.891982,0.636096,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,295097,0.05,7.3,104
2,3,content_4a6607efcb46,57.105568,0.638884,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,128068,0.01,2.2,104
3,4,content_b28d1efd668f,57.094387,0.597811,HOLD,INSUFFICIENT_SIGNAL,286608,0.06,26.2,104
4,5,content_8451fc6f034d,55.930429,0.588047,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,272144,0.03,2.3,20
5,6,content_813e88069237,55.850218,0.594466,HOLD,INSUFFICIENT_SIGNAL,233561,0.06,26.2,104
6,7,content_c1fe78bc4e37,55.759785,0.621414,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,134055,0.03,7.5,104
7,8,content_b115f7c74779,55.165740,0.619107,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,123469,0.03,8.0,104
8,9,content_91652435f57a,55.033483,0.604393,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,159590,0.06,7.8,104
9,10,content_e752a4e03dd3,54.581035,0.609510,HOLD,INSUFFICIENT_SIGNAL,130892,0.01,23.9,104


## 2. Intended use and limits

### Intended use

This playbook is for a **content editor, SEO analyst, or strategist** who needs a manageable review queue. The intended workflow is:

**measured page signals → ranked review queue → human inspection → chosen content action → later measurement**

The queue helps decide **where to spend review time first**. It does not decide the final edit.

### Cost / value thinking

Review capacity is limited. A false positive costs analyst/editor time and may create unnecessary content churn. A false negative can leave a real opportunity unreviewed. I therefore prioritize pages with both measured visibility and stronger review signals, while keeping low-evidence pages in `HOLD`.

The `review_value_score` combines the model's exploratory priority score with log-scaled impressions. It is an operational triage score, not a financial ROI estimate.

### Limits

- The starter file is one trailing-90-day snapshot, not a clean intervention study.
- Week-6 grouped validation showed weak overall discrimination, so the score should not be presented as reliable prediction for unseen clients.
- CTR is affected by rank, query intent, SERP features, brand demand, and measurement context.
- `days_since_last_update` measures staleness, not content quality.
- No action in this notebook establishes that a refresh **causes** recovery.
- Low-volume or missing-position pages may be under-prioritized because there is less measurable evidence.


In [2]:
# Practical capacity / cost-value view.
capacity_rows = []
for k in [10, 20, 50, 100, 250]:
    topk = queue.head(k)
    capacity_rows.append({
        "review_capacity": k,
        "median_impressions": float(topk["impressions_90d"].median()),
        "mean_review_value_score": float(topk["review_value_score"].mean()),
        "observed_decline_proxy_share": float(topk["decline_proxy"].mean()),  # retrospective check only
        "low_ctr_visible_share": float((topk["reason_code"] == "LOW_CTR_VISIBLE").mean()),
    })

capacity = pd.DataFrame(capacity_rows)
print("Capacity / value view (proxy share is retrospective evaluation, not an input):")
display(
    capacity.style.format({
        "median_impressions": "{:,.0f}",
        "mean_review_value_score": "{:.1f}",
        "observed_decline_proxy_share": "{:.1%}",
        "low_ctr_visible_share": "{:.1%}",
    })
)


Capacity / value view (proxy share is retrospective evaluation, not an input):


,review_capacity,median_impressions,mean_review_value_score,observed_decline_proxy_share,low_ctr_visible_share
0,10,"184,134",57.0,40.0%,70.0%
1,20,"144,535",55.0,55.0%,65.0%
2,50,"117,346",52.3,58.0%,68.0%
3,100,"83,872",50.2,46.0%,68.0%
4,250,"53,814",47.7,50.8%,71.6%


## 3. Human review + the no-go list

### Human-review rules

Before acting on any `REVIEW_TITLE_SNIPPET` or `REVIEW_REFRESH` row, a reviewer should check:

1. **Search intent:** Is the page still the correct answer for the queries/users it attracts?
2. **SERP context:** Could low CTR be explained by ads, answer boxes, local packs, branded results, or rank?
3. **Content accuracy/freshness:** Is any factual or product information actually stale?
4. **Cannibalization:** Is another page competing for the same intent?
5. **Business importance:** Does this page support a meaningful journey or conversion, not just traffic?
6. **Recent changes:** Was the page recently edited, migrated, redirected, or affected by tracking changes?
7. **Measurement sufficiency:** Is there enough volume/history to justify action?

### What should NOT be automated

This notebook should **not automatically**:

- publish or rewrite page copy;
- change titles/meta descriptions without review;
- delete, prune, noindex, redirect, or canonicalize pages;
- merge pages;
- alter internal links in production;
- change schema or structured data;
- send client-facing claims that traffic will increase;
- interpret low CTR as proof of poor content;
- treat `REVIEW_REFRESH` as an instruction that a refresh will cause recovery.

The only safe automatic behavior here is **creating a review queue and logging the reason code**. A person owns the final decision.


In [3]:
# Generate a compact reviewer checklist for the top 10 without private URLs/queries.
top10_review = queue.head(10)[[
    "rank", "content_id", "action_label", "reason_code",
    "review_value_score", "impressions_90d", "ctr",
    "avg_position", "days_since_last_update"
]].copy()

def reviewer_question(row):
    if row["reason_code"] == "LOW_CTR_VISIBLE":
        return "Check query intent + SERP layout first; low CTR may be normal at this position/query mix."
    if row["reason_code"] == "STALE_HIGH_VOLUME":
        return "Check whether facts/coverage are actually stale and whether the page was intentionally left unchanged."
    if row["reason_code"] == "VISIBLE_MONITOR":
        return "Confirm there is a real problem before editing; visibility alone is not a refresh reason."
    return "Collect more evidence before acting."

top10_review["human_check"] = [reviewer_question(r) for _, r in top10_review.iterrows()]
display(top10_review)

print("Human review is mandatory for every action above.")


,rank,content_id,action_label,reason_code,review_value_score,impressions_90d,ctr,avg_position,days_since_last_update,human_check
0,1,content_c8e9d6ab9013,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,62.204363,208678,0.00,9.7,104,Check query intent + SERP layout first; low CT...
1,2,content_36ff89c8214e,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,60.891982,295097,0.05,7.3,104,Check query intent + SERP layout first; low CT...
2,3,content_4a6607efcb46,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,57.105568,128068,0.01,2.2,104,Check query intent + SERP layout first; low CT...
3,4,content_b28d1efd668f,HOLD,INSUFFICIENT_SIGNAL,57.094387,286608,0.06,26.2,104,Collect more evidence before acting.
4,5,content_8451fc6f034d,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,55.930429,272144,0.03,2.3,20,Check query intent + SERP layout first; low CT...
5,6,content_813e88069237,HOLD,INSUFFICIENT_SIGNAL,55.850218,233561,0.06,26.2,104,Collect more evidence before acting.
6,7,content_c1fe78bc4e37,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,55.759785,134055,0.03,7.5,104,Check query intent + SERP layout first; low CT...
7,8,content_b115f7c74779,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,55.165740,123469,0.03,8.0,104,Check query intent + SERP layout first; low CT...
8,9,content_91652435f57a,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,55.033483,159590,0.06,7.8,104,Check query intent + SERP layout first; low CT...
9,10,content_e752a4e03dd3,HOLD,INSUFFICIENT_SIGNAL,54.581035,130892,0.01,23.9,104,Collect more evidence before acting.


Human review is mandatory for every action above.


## 4. Monitoring / retrain triggers

This is a **non-production** playbook, but I would monitor it as if it were going to be reused.

### Light monitoring

Recalculate the queue on a regular cadence (for example monthly) and record:

- number/share of rows in each reason code;
- base rate of the evaluation proxy when a later outcome is available;
- Precision@20 and Precision@50 on a future labeled window;
- score distribution;
- missingness / availability of the five model features;
- review acceptance rate: how often editors agree a top-ranked page deserves action;
- action outcomes separately from prediction outcomes.

### Retrain / redesign triggers

I would retrain or reconsider the model when any of these occur:

- **Precision@20 drops below the transparent Week-4 baseline** on the same future evaluation set.
- **AUROC remains near 0.50** or Average Precision is not meaningfully above the future base rate.
- Feature distributions shift materially (for example median impressions or CTR moves by ~25%+).
- Missingness in a required feature changes by 10 percentage points or more.
- A large share of new clients has history unlike the training portfolio.
- Editors reject more than half of top-20 recommendations for two review cycles.
- The meaning or calculation of a source field changes.
- A better time-aware label becomes available; in that case, redesigning the target is more important than simply retraining.

No trigger automatically deploys a new model. It starts a **human validation review**.


In [4]:
# Monitoring thresholds saved as a small receipt for the paper.
monitoring_receipt = {
    "lane": "Refresh / Content Opportunity Scoring",
    "intended_use": "Human-reviewed prioritization only",
    "primary_metric": "Precision@20",
    "retrain_or_redesign_triggers": {
        "p20_below_week4_baseline": True,
        "auroc_near_chance": 0.52,
        "feature_median_shift_fraction": 0.25,
        "missingness_change_percentage_points": 10,
        "editor_top20_rejection_rate": 0.50,
        "rejection_cycles": 2,
    },
    "no_go_automation": [
        "auto-publish",
        "auto-delete/noindex/redirect",
        "automatic causal claims",
        "automatic metadata changes",
    ],
}

display(pd.json_normalize(monitoring_receipt["retrain_or_redesign_triggers"]))


,p20_below_week4_baseline,auroc_near_chance,feature_median_shift_fraction,missingness_change_percentage_points,editor_top20_rejection_rate,rejection_cycles
0,True,0.52,0.25,10,0.5,2


## 5. Exports for the paper

The notebook exports the paper-ready artifacts below:

- `work/outputs/action_playbook_queue.csv` — full ranked review queue. **Regenerated by the notebook; keep out of Git if the repo leak-guard blocks CSVs.**
- `work/outputs/w07_playbook_metrics.json` — small metrics/monitoring receipt; safe to commit.
- `work/figures/w07_action_mix.png` — action/reason-code mix for reuse in the paper.
- `work/figures/w07_staleness_by_action.png` — measured staleness distribution by action archetype.

The queue excludes `trend_direction`, `trend_pct`, and `decline_proxy`. Outcome information is used only for retrospective notebook checks and metrics, not for ranking or exported recommendations.


In [5]:
import matplotlib.pyplot as plt

out_dir = REPO_ROOT / "work/outputs"
fig_dir = REPO_ROOT / "work/figures"
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

export_cols = [
    "rank",
    "content_id",
    "client_id",
    "review_value_score",
    "review_priority_score",
    "action_label",
    "reason_code",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
]
queue_export = queue[export_cols].copy()

forbidden_exports = {"trend_direction", "trend_pct", "decline_proxy"}
assert forbidden_exports.isdisjoint(queue_export.columns)

queue_path = out_dir / "action_playbook_queue.csv"
queue_export.to_csv(queue_path, index=False)

metrics = {
    "rows_ranked": int(len(queue_export)),
    "action_counts": {k: int(v) for k, v in queue_export["action_label"].value_counts().items()},
    "reason_counts": {k: int(v) for k, v in queue_export["reason_code"].value_counts().items()},
    "top20_observed_decline_proxy_share_retrospective_only": float(queue.head(20)["decline_proxy"].mean()),
    "top50_observed_decline_proxy_share_retrospective_only": float(queue.head(50)["decline_proxy"].mean()),
    "week6_validation_note": (
        "Grouped-client validation showed limited overall discrimination; "
        "playbook is exploratory human-reviewed decision-support."
    ),
    "monitoring": monitoring_receipt,
}
metrics_path = out_dir / "w07_playbook_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))

# Figure 1: action mix
action_counts = queue_export["action_label"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(8, 4.8))
action_counts.plot(kind="bar")
plt.title("Action playbook mix")
plt.xlabel("Action")
plt.ylabel("Pages")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
action_fig = fig_dir / "w07_action_mix.png"
plt.savefig(action_fig, dpi=160)
plt.close()

# Figure 2: median days since update by action archetype
stale_summary = (
    queue_export.groupby("action_label", observed=True)["days_since_last_update"]
    .median()
    .sort_values(ascending=False)
)
plt.figure(figsize=(8, 4.8))
stale_summary.plot(kind="bar")
plt.title("Median days since last update by action")
plt.xlabel("Action")
plt.ylabel("Median days since last update")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
stale_fig = fig_dir / "w07_staleness_by_action.png"
plt.savefig(stale_fig, dpi=160)
plt.close()

print("Exports written:")
print(queue_path)
print(metrics_path)
print(action_fig)
print(stale_fig)
print("\nQueue preview:")
display(queue_export.head(10))


Exports written:
/mnt/data/w07repo/flyrank-ML-internship-main/work/outputs/action_playbook_queue.csv
/mnt/data/w07repo/flyrank-ML-internship-main/work/outputs/w07_playbook_metrics.json
/mnt/data/w07repo/flyrank-ML-internship-main/work/figures/w07_action_mix.png
/mnt/data/w07repo/flyrank-ML-internship-main/work/figures/w07_staleness_by_action.png

Queue preview:


,rank,content_id,client_id,review_value_score,review_priority_score,action_label,reason_code,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update
0,1,content_c8e9d6ab9013,client_19581e27de,62.204363,0.668188,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,208678,0,0.00,9.7,104
1,2,content_36ff89c8214e,client_19581e27de,60.891982,0.636096,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,295097,154,0.05,7.3,104
2,3,content_4a6607efcb46,client_6208ef0f77,57.105568,0.638884,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,128068,17,0.01,2.2,104
3,4,content_b28d1efd668f,client_6208ef0f77,57.094387,0.597811,HOLD,INSUFFICIENT_SIGNAL,286608,169,0.06,26.2,104
4,5,content_8451fc6f034d,client_d029fa3a95,55.930429,0.588047,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,272144,75,0.03,2.3,20
5,6,content_813e88069237,client_6208ef0f77,55.850218,0.594466,HOLD,INSUFFICIENT_SIGNAL,233561,129,0.06,26.2,104
6,7,content_c1fe78bc4e37,client_19581e27de,55.759785,0.621414,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,134055,43,0.03,7.5,104
7,8,content_b115f7c74779,client_19581e27de,55.165740,0.619107,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,123469,37,0.03,8.0,104
8,9,content_91652435f57a,client_19581e27de,55.033483,0.604393,REVIEW_TITLE_SNIPPET,LOW_CTR_VISIBLE,159590,100,0.06,7.8,104
9,10,content_e752a4e03dd3,client_6208ef0f77,54.581035,0.609510,HOLD,INSUFFICIENT_SIGNAL,130892,15,0.01,23.9,104


## Self-check

- [x] Ranked actions have one primary reason code and one action label per page.
- [x] Archetype → action mapping is stated in plain language.
- [x] The decay/refresh insight is framed as observed/directional, not causal.
- [x] Intended use is human-reviewed prioritization.
- [x] Limits include the snapshot design and weak grouped-client model discrimination.
- [x] Cost/value thinking distinguishes wasted review time from missed opportunities.
- [x] Human-review rules are explicit.
- [x] The no-go list clearly states what must NOT be automated.
- [x] Monitoring and retrain/redesign triggers are practical and light.
- [x] No trigger automatically deploys a model; it starts human validation.
- [x] `work/outputs/action_playbook_queue.csv` is regenerated by the notebook.
- [x] `work/outputs/w07_playbook_metrics.json` is generated as a paper receipt.
- [x] Two reusable paper figures are written to `work/figures/`.
- [x] Exported recommendation queue contains no `trend_direction`, `trend_pct`, or label-derived proxy.
- [x] No client names, URLs, private queries, or secrets are included.
- [x] Notebook runs top to bottom with no errors.
- [ ] Commit `work/notebooks/w07_action_playbook.ipynb`, the metrics JSON, and reusable figures; keep the queue CSV out of Git if blocked by the repo's leak guard.
